## 실습 2: Memory를 추가하여 Agent 개인화하기

### 개요

실습 1에서는 로컬 세션에서 단일 사용자를 대상으로 원활하게 작동하는 고객 지원 Agent를 구축했습니다. 하지만 실제 고객 지원 환경에서는 로컬 환경의 단일 사용자를 넘어 확장할 수 있어야 합니다.

**프로덕션 환경에서 Agent**를 실행하려면 다음 기능이 필요합니다.
- **다중 사용자 지원**: 수천 명의 고객을 동시에 처리
- **영구 스토리지**: 세션 수명 주기가 끝난 후에도 대화 저장
- **장기 학습**: 고객 선호도 및 행동 패턴 추출
- **세션 간 연속성**: 서로 다른 상호 작용에서도 고객 정보 기억

**워크숍 진행 상황:**
- **실습 1(완료)**: Agent 프로토타입 만들기 - 작동하는 고객 지원 Agent 구축
- **실습 2(현재)**: Memory로 기능 강화 - 대화 컨텍스트 및 개인화 추가
- **실습 3**: Gateway 및 Identity로 확장 - Agent 간에 도구를 안전하게 공유
- **실습 4**: 프로덕션에 배포 - AgentCore Runtime 및 Observability 사용
- **실습 5**: Agent 성능 평가 - 온라인 평가를 통해 품질 모니터링
- **실습 6**: 사용자 인터페이스 구축 - 고객용 애플리케이션 만들기

이 실습에서는 대화를 금방 잊어버리는 Agent를 지능형 맞춤형 Assistant로 전환하는 데 필요한 영속성 및 학습 계층을 추가합니다.

Memory는 지능을 구성하는 핵심 요소입니다. Large Language Models(LLM)는 뛰어난 기능을 제공하지만 대화 간 정보를 유지하는 영구 Memory가 없습니다. [Amazon Bedrock AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-getting-started.html)는 AI Agent가 시간 경과에 따라 컨텍스트를 유지하고 중요한 사실을 기억하며 일관된 맞춤형 경험을 제공하도록 지원하는 관리형 서비스를 통해 이러한 한계를 해결합니다.

AgentCore Memory는 두 가지 수준으로 작동합니다.
- **Short-Term Memory**: 단일 상호 작용이나 밀접하게 관련된 세션 내에서 연속성을 제공하는 즉각적인 대화 컨텍스트 및 세션 기반 정보입니다.
- **Long-Term Memory**: 여러 대화에서 추출하여 저장하는 영구 정보입니다. 시간이 지나도 맞춤형 경험을 제공할 수 있도록 사실, 선호도, 요약 등을 포함합니다.

### 실습 2 아키텍처
<div style="text-align:left">
    <img src="images/architecture_lab2_memory.png" width="75%"/>
</div>

*영구 Short-Term Memory 및 Long-Term Memory 기능을 갖춘 다중 사용자 Agent입니다. *

### 사전 요구 사항

* 적절한 권한이 있는 **AWS 계정**
* 로컬에 설치된 **Python 3.10 이상**
* 자격 증명으로 구성된 **AWS CLI**
* [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)에서 활성화된 **Amazon Nova 2 Lite**
* 다음 셀에서 설치할 **Strands Agents** 및 기타 라이브러리
* AWS 워크숍 계정에는 다음 리소스가 미리 생성되어 있습니다.
    - AWS Lambda 함수 
    - AWS Lambda 실행 IAM Role
    - AgentCore Gateway IAM Role
    - AWS Lambda 함수에서 사용하는 DynamoDB 테이블 
    - Cognito User Pool 및 User Pool Client


### 단계 1: 라이브러리 가져오기

AgentCore Memory에 필요한 라이브러리를 가져옵니다. 여기서는 AgentCore 기능을 손쉽게 사용할 수 있도록 지원하는 경량 래퍼인 [Amazon Bedrock AgentCore Python SDK](https://github.com/aws/bedrock-agentcore-sdk-python)를 사용합니다.

In [ ]:
import logging
from boto3.session import Session

from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from lab_helpers.utils import put_ssm_parameter

boto_session = Session()
REGION = boto_session.region_name

logger = logging.getLogger(__name__)

### 단계 2: Bedrock AgentCore Memory 리소스 생성

Amazon Bedrock AgentCore Memory는 AI Agent에 영구 Memory 기능을 제공하는 완전관리형 서비스입니다.

#### AgentCore Memory 개념

1. **Short-Term Memory(STM)**: 세션 내의 대화 컨텍스트를 즉시 저장합니다.
2. **Long-Term Memory(LTM)**: STM을 비동기식으로 처리하여 의미 있는 패턴, 선호도, 사실을 추출합니다.
3. **Memory 전략**: 정보를 추출하고 구성하는 다양한 방식입니다.
   - **USER_PREFERENCE**: 고객 선호도, 행동, 패턴을 학습합니다.
   - **SEMANTIC**: 유사도 검색을 위해 벡터 임베딩을 사용하여 사실 정보를 저장합니다.
4. **네임스페이스**: 고객 및 컨텍스트 유형별로 Memory를 논리적으로 그룹화합니다. 다음 두 네임스페이스를 생성합니다.
- `support/customer/{actorId}/preferences/`: 고객 선호도 및 행동 패턴
- `support/customer/{actorId}/semantic/`: 사실 정보 및 대화 기록

이 구조는 각 고객의 정보를 격리하고 쉽게 검색할 수 있는 멀티 테넌트 Memory를 지원합니다.

#### Memory 생성 과정

Memory 리소스를 생성하려면 기반 인프라(벡터 데이터베이스, 처리 파이프라인 등)를 프로비저닝해야 합니다. AWS가 내부적으로 관리형 서비스를 설정하므로 일반적으로 2~3분이 걸립니다.

In [ ]:
memory_name = "CustomerSupportMemory"

memory_manager = MemoryManager(region_name=REGION)
memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "CustomerPreferences",
                "description": "Captures customer preferences and behavior",
                "namespaces": ["support/customer/{actorId}/preferences/"],
            }
        },
        {
            StrategyType.SEMANTIC.value: {
                "name": "CustomerSupportSemantic",
                "description": "Stores facts from conversations",
                "namespaces": ["support/customer/{actorId}/semantic/"],
            }
        },
    ],
)
memory_id = memory["id"]
put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)

In [ ]:
if memory_id:
    print("✅ AgentCore Memory created successfully!")
    print(f"Memory ID: {memory_id}")
else:
    print("Memory resource not created. Try Again !")

## 단계 3: 이전 고객 상호 작용 시드 데이터 추가

**Memory에 시드 데이터를 추가하는 이유는 무엇인가요?**

프로덕션 환경에서는 Agent가 고객과 상호 작용하면서 자연스럽게 Memory를 축적합니다. 하지만 이 실습에서는 실제 대화를 기다리지 않고 Long-Term Memory(LTM)의 작동 방식을 살펴보기 위해 이전 대화 데이터를 미리 추가합니다.

**Memory 처리 방식:**
1. `create_event`는 상호 작용을 **Short-Term Memory**(STM)에 즉시 저장합니다.
2. STM은 **Long-Term Memory** 전략을 통해 비동기식으로 처리됩니다.
3. LTM은 나중에 검색할 수 있도록 패턴, 선호도, 사실을 추출합니다.

고객 이력 데이터를 추가하여 실제 작동 방식을 확인해 보겠습니다.

In [ ]:
from lab_helpers.lab2_memory import ACTOR_ID


# 이전 고객 상호 작용을 시드 데이터로 추가
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    (
        "I can help with that thermal issue. For video editing workloads, let's check your Activity Monitor and adjust performance settings. Your MacBook Pro order #MB-78432 is still under warranty.",
        "ASSISTANT",
    ),
    (
        "What's the return policy on gaming headphones? I need low latency for competitive FPS games",
        "USER",
    ),
    (
        "For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking the audio latency specs - most gaming models have <40ms latency.",
        "ASSISTANT",
    ),
    (
        "I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.",
        "USER",
    ),
    (
        "Perfect! For development work, I'd suggest looking at our ThinkPad E series or Dell XPS models. Both have excellent Linux support and 16GB RAM options within your budget.",
        "ASSISTANT",
    ),
]

# 이전 상호 작용 저장
if memory_id:
    try:
        memory_client = MemoryClient(region_name=REGION)
        memory_client.create_event(
            memory_id=memory_id,
            actor_id=ACTOR_ID,
            session_id="previous_session",
            messages=previous_interactions,
        )
        print("✅ Seeded customer history successfully")
        print("📝 Interactions saved to Short-Term Memory")
        print("⏳ Long-Term Memory processing will begin automatically...")
    except Exception as e:
        print(f"⚠️ Error seeding history: {e}")

### Memory 처리 방식 이해

`create_event`로 이벤트를 생성하면 AgentCore Memory가 데이터를 두 단계로 처리합니다.

1. **즉시 처리**: 메시지를 Short-Term Memory(STM)에 저장합니다.
2. **비동기 처리**: STM을 Long-Term Memory(LTM) 전략으로 처리합니다.

시스템에서 다음 작업을 수행하므로 LTM 처리에는 일반적으로 20~30초가 걸립니다.
- 대화 패턴 분석
- 고객 선호도 및 행동 추출
- 사실 정보에 대한 시맨틱 임베딩 생성
- 효율적인 검색을 위해 네임스페이스별로 Memory 구성

고객 선호도를 검색하여 Long-Term Memory 처리가 완료되었는지 확인합니다.

In [ ]:
import time

# Long-Term Memory 처리가 완료될 때까지 대기
print("🔍 Checking for processed Long-Term Memories...")
retries = 0
max_retries = 6  # 1분 대기

while retries < max_retries:
    memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"support/customer/{ACTOR_ID}/preferences/",
        query="can you summarize the support issue",
    )

    if memories:
        print(f"✅ Found {len(memories)} preference memories after {retries * 10} seconds!")
        break

    retries += 1
    if retries < max_retries:
        print(f"⏳ Still processing... waiting 10 more seconds (attempt {retries}/{max_retries})")
        time.sleep(10)
    else:
        print("⚠️ Memory processing is taking longer than expected. This can happen with overloading..")
        break

print("🎯 AgentCore Memory automatically extracted these customer preferences from our seeded conversations:")
print("=" * 80)

for i, memory in enumerate(memories, 1):
    if isinstance(memory, dict):
        content = memory.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            print(f"  {i}. {text}")

### Semantic Memory 살펴보기

Semantic Memory는 벡터 임베딩을 사용하여 대화에서 얻은 사실 정보를 저장합니다. 이를 통해 관련 사실과 컨텍스트를 유사도 기반으로 검색할 수 있습니다.

In [ ]:
import time

# Semantic Memory 검색(사실 정보)
while True:
    semantic_memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"support/customer/{ACTOR_ID}/semantic/",
        query="information on the technical support issue",
    )
    print("🧠 AgentCore Memory identified these factual details from conversations:")
    print("=" * 80)
    if semantic_memories:
        break
    time.sleep(10)
for i, memory in enumerate(semantic_memories, 1):
    if isinstance(memory, dict):
        content = memory.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            print(f"  {i}. {text}")

## 단계 4: Memory를 사용하는 고객 지원 Agent 생성

다음으로 실습 1과 동일하게 고객 지원 Agent를 구현하되, 이번에는 Agent의 세션 관리자에 AgentCoreMemoryConfig를 추가합니다. 이를 통해 Agent가 새 이벤트를 생성하고 Memory를 검색할 수 있습니다.

In [ ]:
import uuid

from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.memory.integrations.strands.config import (
    AgentCoreMemoryConfig,
    RetrievalConfig,
)
from bedrock_agentcore.memory.integrations.strands.session_manager import (
    AgentCoreMemorySessionManager,
)

from lab_helpers.lab1_strands_agent import (
    SYSTEM_PROMPT,
    get_return_policy,
    web_search,
    get_product_info,
    get_technical_support,
    MODEL_ID,
)

session_id = uuid.uuid4()

memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=str(session_id),
    actor_id=ACTOR_ID,
    retrieval_config={
        "support/customer/{actorId}/semantic/": RetrievalConfig(top_k=3, relevance_score=0.2),
        "support/customer/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.2),
    },
)

# Bedrock 모델 초기화(Amazon Nova 2 Lite)
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

# 다섯 가지 도구를 모두 포함한 고객 지원 Agent 생성
agent = Agent(
    model=model,
    session_manager=AgentCoreMemorySessionManager(memory_config, REGION),
    tools=[
        get_product_info,  # Tool 1: 간단한 제품 정보 조회
        get_return_policy,  # Tool 2: 간단한 반품 policy 조회
        web_search,
        get_technical_support,
    ],
    system_prompt=SYSTEM_PROMPT,
)

## 단계 5: 개인화된 Agent 테스트

Memory로 기능을 강화한 Agent를 테스트해 보겠습니다. 고객의 이전 선호도를 사용하여 맞춤형 추천을 제공하는 방식을 확인하세요.

Agent는 다음 작업을 자동으로 수행합니다.
1. Memory에서 관련 고객 컨텍스트 검색
2. 해당 컨텍스트를 사용하여 응답 개인화
3. 나중에 사용할 수 있도록 새로운 상호 작용 저장

In [ ]:
print("🎧 Testing headphone recommendation with customer memory...\n\n")
response1 = agent("Which headphones would you recommend?")

In [ ]:
print("\n💻 Testing laptop preference recall...\n\n")
response2 = agent("What is my preferred laptop brand and requirements?")

Agent가 다음 정보를 어떻게 기억하는지 확인하세요.

- 게임 관련 선호도(낮은 지연 시간의 헤드폰)
- 노트북 선호도(ThinkPad, 16GB RAM, Linux 호환성)
- 예산 제약(노트북 구매 예산 $1200)
- 이전 기술 문제(MacBook 과열) 

영속적이고 개인화된 고객 경험을 제공하는 것이 바로 AgentCore Memory의 강점입니다!

## 축하합니다! 🎉

**실습 2: 고객 지원 Agent에 Memory 추가**를 성공적으로 완료했습니다!

### 완료한 작업

- Amazon Bedrock AgentCore Memory를 사용하여 서버리스 관리형 Memory 생성
- 사용자 선호도 및 Semantic(사실) 정보를 저장하는 Long-Term Memory 구현
- Strands Agents에서 제공하는 세션 관리 메커니즘을 사용하여 AgentCore Memory와 고객 지원 Agent 통합

##### 다음 실습: [실습 3 - Gateway 및 Identity를 사용한 확장 →](lab-03-agentcore-gateway.ipynb)

## 리소스
- [Amazon Bedrock Agent Core Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)
- [Amazon Bedrock AgentCore Memory 심층 분석 블로그](https://aws.amazon.com/blogs/machine-learning/amazon-bedrock-agentcore-memory-building-context-aware-agents/)
- [Strands Agent SDK - AgentCore Memory 예제](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/strands-sdk-memory.html)